# A. Importation of libraries and Configs

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    # Path to the pseudonymized revenues dataset
    dataset_dir     = r"Database\revenues_pseudonymized.xlsx",
    # Path to the enrollee infos
    enrollees_dir   = r"Database\enrollees_pseudonymized.xlsx",
    # Path to the machine learning model parameters
    parameters_dir  = r"machine_learning\parameters.json",

    # Path to cache directory to store preprocessed dataset if needed
    cache_dir       = "",
    load_cache      = True,

    # Path to store results
    results_dir     = r"C:\Users\rjbel\Python\Data\Thesis\Results",

    # The date used
    observation_end = datetime.today(),

    # Class to predict
    target_feature  = 'dtp_bracket',
    # Test size in %
    test_size       = 0.3,

    # Time points used in generating survival features
    time_points     = [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 390, 420, 450],
)


# B. Loading of datasets

## 1. Revenues

In [ ]:
df_revenues = pd.read_excel(args.dataset_dir)

In [ ]:
df_revenues

## 2. Enrollees

In [ ]:
df_enrollees = pd.read_excel(args.enrollees_dir)

In [ ]:
df_enrollees

## 3. Credit Sales

In [ ]:
from feature_engineering.credit_sales_machine_learning import CreditSalesProcessor

cs = CreditSalesProcessor(df_revenues, df_enrollees, args)
df_credit_sales = cs.show_data()

In [ ]:
df_credit_sales.to_excel('credit_sale_invoices.xlsx', index=False)

In [ ]:
df_credit_sales

# C. Exploratory Data Analysis

In [ ]:
# Get counts
counts = df_credit_sales.dtp_bracket.value_counts()

# Convert to percentages
percentages = counts / counts.sum() * 100

# Combine into one DataFrame
result = pd.DataFrame({
    'count': counts,
    'percentage': percentages.round(2)
})

print(result)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Remove those that have no full dtp_history:
df_filtered = df_credit_sales.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'])


# Clean and filter directly on the DataFrame
df_filtered = df_filtered.loc[
    df_filtered['days_elapsed_until_fully_paid']
        .replace("", np.nan)   # replace empty strings with NaN
        .dropna()              # drop NaNs
        .index                 # keep aligned index
]


# Apply numeric filter
df_filtered = df_filtered[
    (df_filtered['days_elapsed_until_fully_paid'] >= -200) &
    (df_filtered['days_elapsed_until_fully_paid'] <= 200)
]

# Convert censor column to categorical with labels
df_filtered["censor_label"] = (
    df_filtered["censor"]
    .map({0: "Still Unpaid", 1: "Fully Paid"})
    .astype("category")   # force categorical type
)


# KDE plot with grouping by categorical censor labels
sns.kdeplot(
    data=df_filtered,
    x="days_elapsed_until_fully_paid",
    hue="censor_label",
    fill=False,
    common_norm=False,
    palette="Set1"
)

plt.title("KDE Plot: Days Elapsed Until Fully Paid (-200 to +200)")
plt.xlabel("Days Elapsed")
plt.ylabel("Density")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_filtered = df_credit_sales[df_credit_sales['censor'] == 1]
df_filtered = df_filtered.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'])

# Select relevant columns
cols = ['days_elapsed_until_fully_paid', 
        'dtp_1', 'dtp_2', 'dtp_3', 'dtp_4', 
        'dtp_avg', 'dtp_wavg', 'dtp_2_trend',
        'dtp_3_trend', 'days_since_last_payment',
        'credit_sale_amount', 'amount_due_cumsum',
        'amount_paid_cumsum', 'opening_balance']

# Compute correlation matrix
corr = df_filtered[cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation with Days Elapsed Until Fully Paid")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_filtered = df_credit_sales[df_credit_sales['censor'] == 1]
df_filtered = df_filtered.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'])

# Select relevant columns
cols = ['days_elapsed_until_fully_paid', 'opening_balance_flag', 'payment_ratio',
        'dtp_rolling_std', 'dtp_max', 'early_payer_flag',
        'due_month', 'due_quarter']

# Compute correlation matrix
corr = df_filtered[cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation with Days Elapsed Until Fully Paid")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Filter and clean data
df_filtered = df_credit_sales[df_credit_sales['censor'] == 1]
df_filtered = df_filtered.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'])

# Define groups of columns
groups = {
    "Payment Dynamics": [
        'days_elapsed_until_fully_paid', 'days_since_last_payment',
        'payment_ratio', 'early_payer_flag'
    ],
    "DTP Basics": ['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'],
    "DTP Aggregates": ['dtp_avg', 'dtp_wavg'],
    "DTP Trends": ['dtp_2_trend', 'dtp_3_trend'],
    "DTP Variability": ['dtp_rolling_std', 'dtp_max'],
    "Financial Amounts": [
        'credit_sale_amount', 'amount_due_cumsum',
        'amount_paid_cumsum', 'opening_balance',
        'opening_balance_flag'
    ],
    "Calendar Features": ['due_month', 'due_quarter']
}

def pairgrid_with_heatmap(df, cols, hue, title):
    # Build grid
    g = sns.PairGrid(df[cols + [hue]], hue=hue, diag_sharey=False)

    # Lower triangle: scatterplots
    g.map_lower(sns.scatterplot, alpha=0.6, s=40, edgecolor='k')

    # Diagonal: KDE plots
    g.map_diag(sns.kdeplot, fill=True)

    # Upper triangle: correlation heatmap
    corr = df[cols].corr()
    for i, col_i in enumerate(cols):
        for j, col_j in enumerate(cols):
            if j > i:  # upper triangle only
                ax = g.axes[i, j]
                c = corr.loc[col_i, col_j]
                # Map correlation [-1,1] to colormap
                ax.set_facecolor(plt.colormaps['coolwarm']((c+1)/2))
                ax.text(0.5, 0.5, f"{c:.2f}", ha='center', va='center', fontsize=12)
                ax.set_xticks([])
                ax.set_yticks([])

    g.add_legend()
    plt.suptitle(title, y=1.02)
    plt.show()

# Generate plots for each group
for group_name, cols in groups.items():
    pairgrid_with_heatmap(df_filtered, cols, 'dtp_bracket', f"{group_name} Features")

### c. Linear Discriminant Analysis

In [ ]:
df_credit_sales.columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
BRACKET_COL  = 'dtp_bracket'

DTP_FEATURES = [
    'dtp_1', 'dtp_2', 'dtp_3', 'dtp_4',
    'dtp_avg', 'dtp_wavg',
    'dtp_rolling_std', 'dtp_max',
    'dtp_2_trend', 'dtp_3_trend'
]

FIN_FEATURES = [
    'credit_sale_amount', 'opening_balance',
    'amount_due_cumsum', 'amount_paid_cumsum'
]

BEHAVIOUR_FEATURES = [
    'payment_ratio', 'opening_balance_flag',
    'early_payer_flag', 'days_since_last_payment',
    'on_time_streak', 'prev_bracket'
]

TEMPORAL_FEATURES = ['due_month', 'due_quarter']

PLAN_FEATURES = [
    'plan_type_risk_score',
    'plan_type_Plan - A', 'plan_type_Plan - B',
    'plan_type_Plan - C', 'plan_type_Plan - D',
    'plan_type_Plan - E', 'plan_type_nan'
]

ALL_FEATURES = (
    DTP_FEATURES +
    FIN_FEATURES +
    BEHAVIOUR_FEATURES +
    TEMPORAL_FEATURES +
    PLAN_FEATURES
)

BRACKET_ORDER  = ['on_time', '30_days', '60_days', '90_days']
BRACKET_COLORS = {
    'on_time': "#382ecc",
    '30_days': "#68e25d",
    '60_days': "#7ab7f0",
    '90_days': '#e07b72',
}


# ─────────────────────────────────────────────
# DATA HELPERS
# ─────────────────────────────────────────────

def load_and_prepare(df):
    """Clean and split into X (float64) and y (bracket label)."""
    cols_needed = ALL_FEATURES + [BRACKET_COL]
    df_clean = df[cols_needed].dropna()
    df_clean = df_clean[df_clean[BRACKET_COL].isin(BRACKET_ORDER)]
    X = df_clean[ALL_FEATURES].astype(float).copy()
    y = df_clean[BRACKET_COL].copy()
    return X, y


def log1p_transform(X):
    """log1p-transform financial features; leave DTP features untouched."""
    X_t = X.astype(float).copy()
    for col in FIN_FEATURES:
        if col in X_t.columns:
            shift = abs(X_t[col].min()) if X_t[col].min() < 0 else 0
            X_t[col] = np.log1p(X_t[col] + shift)
    return X_t


def fit_lda(X, y):
    """Scale then fit LDA; return (pipeline, projected array, explained var ratio)."""
    n_components = min(len(BRACKET_ORDER) - 1, X.shape[1])   # max 3
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lda',    LinearDiscriminantAnalysis(n_components=n_components,
                                              solver='svd')),
    ])
    X_lda = pipe.fit_transform(X, y)
    evr   = pipe.named_steps['lda'].explained_variance_ratio_
    return pipe, X_lda, evr


def compute_class_separation(X, y):
    """Fisher between/within variance ratio per feature."""
    rows = []
    for col in X.columns:
        overall_mean = X[col].mean()
        classes      = y.unique()
        between = sum(
            (y == c).sum() * (X.loc[y == c, col].mean() - overall_mean) ** 2
            for c in classes
        ) / len(y)
        within = sum(
            X.loc[y == c, col].var() * (y == c).sum()
            for c in classes
        ) / len(y)
        rows.append({'feature': col,
                     'separation': between / within if within > 0 else 0})
    return pd.DataFrame(rows).sort_values('separation', ascending=False)


# ─────────────────────────────────────────────
# ATOMIC PLOT HELPERS
# ─────────────────────────────────────────────

def _draw_raw_hist(ax, col, y, feat, show_legend=False):
    """Overlapping density histogram for one raw feature."""
    col = col.astype(float)
    lo, hi = col.quantile(0.01), col.quantile(0.99)
    for bracket in BRACKET_ORDER:
        vals = col[y == bracket].clip(lo, hi)
        ax.hist(vals, bins=60, density=True, alpha=0.55,
                color=BRACKET_COLORS[bracket], label=bracket)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.tick_params(labelsize=7)
    if show_legend:
        ax.legend(fontsize=7, loc='upper right')


def _draw_ld_hist(ax, X_lda, y, evr, ld_idx, show_legend=False):
    """Density histogram for one LDA discriminant axis."""
    for bracket in BRACKET_ORDER:
        mask = (y == bracket).values
        ax.hist(X_lda[mask, ld_idx], bins=80, density=True, alpha=0.55,
                color=BRACKET_COLORS[bracket], label=bracket)
    ax.set_title(f'LD{ld_idx+1}  ({evr[ld_idx]*100:.1f}% sep. var.)',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel(f'LD{ld_idx+1} score', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.tick_params(labelsize=7)
    if show_legend:
        ax.legend(fontsize=7, loc='upper right')


def _draw_ld_scatter(ax, X_lda, y, evr):
    """LD1 vs LD2 2-D scatter coloured by bracket."""
    for bracket in BRACKET_ORDER:
        mask = (y == bracket).values
        ax.scatter(X_lda[mask, 0], X_lda[mask, 1],
                   s=5, alpha=0.28, color=BRACKET_COLORS[bracket],
                   label=bracket, rasterized=True)
    ax.set_xlabel(f'LD1  ({evr[0]*100:.1f}%)', fontsize=8)
    ld2_label = f'LD2  ({evr[1]*100:.1f}%)' if len(evr) > 1 else 'LD2'
    ax.set_ylabel(ld2_label, fontsize=8)
    ax.set_title('LD1 vs LD2 — 2-D projection', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7, markerscale=3)
    ax.tick_params(labelsize=7)


def _col_header(ax, text, color):
    """Solid-background header label above an axis."""
    ax.annotate(text, xy=(0.5, 1.24), xycoords='axes fraction',
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round,pad=0.4', fc=color,
                          alpha=0.85, ec='none'))


def _draw_separation_bar(ax, sep_before, sep_after):
    """Grouped bar chart: Fisher separation before vs after."""
    features = sep_before['feature'].tolist()
    x, w     = np.arange(len(features)), 0.35
    b_vals   = sep_before.set_index('feature').loc[features, 'separation'].values
    a_vals   = sep_after.set_index('feature').loc[features,  'separation'].values
    ax.bar(x - w/2, b_vals, w, label='Before (raw)',         color='#aab7b8')
    ax.bar(x + w/2, a_vals, w, label='After (log1p+scaled)', color='#5dade2')
    ax.set_xticks(x)
    ax.set_xticklabels(features, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel("Fisher's separation ratio", fontsize=9)
    ax.set_title("Per-feature class separation — before vs after log1p transform",
                 fontsize=10)
    ax.legend(fontsize=8)


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def run_lda_analysis(df):
    """
    Two-column layout — one feature per row:

        LEFT  column  =  BEFORE  (raw histogram, p1–p99 clipped)
        RIGHT column  =  AFTER   (LDA component histogram)

    Feature rows:
        Rows 0–5   : dtp_1, dtp_2, dtp_3, dtp_4, dtp_avg, dtp_wavg
        Rows 6–10  : credit_sale_amount, opening_balance,
                     amount_due_cumsum, amount_paid_cumsum, payment_ratio

    Extra rows:
        Row 11 : [note | LD1 vs LD2 scatter]
        Row 12 : Fisher separation bar  (spans both columns)
    """
    # ── data ───────────────────────────────────
    X_raw, y         = load_and_prepare(df)
    X_t              = log1p_transform(X_raw)
    pipe, X_lda, evr = fit_lda(X_t, y)

    # console report
    print(f"\n{'='*55}")
    print("  LDA — Explained separation variance")
    print(f"{'='*55}")
    for i, v in enumerate(evr, 1):
        print(f"  LD{i}: {'█'*int(v*40):<40}  {v*100:.1f}%")
    print(f"{'='*55}\n")

    sep_before = compute_class_separation(X_raw, y)
    sep_after  = compute_class_separation(X_t,   y)
    merged = (sep_before.rename(columns={'separation': 'before'})
                         .merge(sep_after.rename(columns={'separation': 'after'}),
                                on='feature'))
    merged['improvement_%'] = ((merged['after'] - merged['before'])
                               / merged['before'].replace(0, np.nan) * 100)
    print(merged.to_string(index=False))

    # ── grid ───────────────────────────────────
    fin_plus  = FIN_FEATURES + ['payment_ratio']    # 5 features
    all_feats = DTP_FEATURES + fin_plus             # 11 total
    n_feats   = len(all_feats)
    n_ld      = X_lda.shape[1]                      # 3 at most

    # Right column content (after):
    #   Row 0        → LD1 histogram
    #   Row 1        → LD2 histogram
    #   Row 2        → LD3 histogram
    #   Row 3        → LD1 vs LD2 scatter  (spans remaining right cells via axis('off'))
    #   Rows 4–10    → empty (right side blank, left keeps all raw features)
    # Bottom row     → Fisher bar spanning both columns

    NROWS = n_feats + 1    # 11 feature rows + 1 bar row
    NCOLS = 2

    fig, axes = plt.subplots(
        NROWS, NCOLS,
        figsize=(14, NROWS * 2.6),
        gridspec_kw={'hspace': 0.80, 'wspace': 0.28}
    )
    fig.suptitle(
        "LDA Analysis — Before & After  |  dtp_bracket classification",
        fontsize=13, fontweight='bold', y=1.001
    )

    # column header badges on the first row
    _col_header(axes[0, 0], "◀  BEFORE  —  Raw feature distributions", '#922b21')
    _col_header(axes[0, 1], "AFTER  —  LDA discriminant components  ▶", '#1a5276')

    # ── LEFT column: all 11 raw feature histograms ──
    for i, feat in enumerate(all_feats):
        _draw_raw_hist(axes[i, 0], X_raw[feat], y, feat,
                       show_legend=(i == 0))

    # ── RIGHT column ────────────────────────────────
    # Rows 0–(n_ld-1): one LD histogram each
    for ld_idx in range(n_ld):
        _draw_ld_hist(axes[ld_idx, 1], X_lda, y, evr, ld_idx,
                      show_legend=(ld_idx == 0))

    # Row n_ld: LD1 vs LD2 scatter
    _draw_ld_scatter(axes[n_ld, 1], X_lda, y, evr)

    # Rows (n_ld+1) onward on the right: turn off (left keeps raw hists)
    for i in range(n_ld + 1, n_feats):
        axes[i, 1].axis('off')

    # ── Fisher bar row — spans both columns ─────────
    br = n_feats
    for c in range(NCOLS):
        axes[br, c].axis('off')

    pos_l  = axes[br, 0].get_position()
    pos_r  = axes[br, 1].get_position()
    bar_ax = fig.add_axes((
        pos_l.x0,
        pos_l.y0,
        pos_r.x1 - pos_l.x0,
        pos_l.height * 1.8
    ))
    _draw_separation_bar(bar_ax, sep_before, sep_after)

    plt.savefig('Images/lda_before_after.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nSaved → lda_before_after.png")

    return pipe, X_lda, evr, merged


# ─────────────────────────────────────────────
# USAGE
# ─────────────────────────────────────────────
# pipe, X_lda, evr, sep_df = run_lda_analysis(df_credit_sales)
#
# Inject LD components back into your dataframe for modelling:
#   X_raw, y = load_and_prepare(df_credit_sales)
#   X_t      = log1p_transform(X_raw)
#   comps    = pipe.transform(X_t)
#   for i in range(comps.shape[1]):
#       df_credit_sales.loc[X_raw.index, f'LD{i+1}'] = comps[:, i]

In [ ]:
df_credit_sales.columns

In [ ]:
run_lda_analysis(df_credit_sales)

### d. LDA (Delinquent)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
BRACKET_COL  = 'dtp_bracket'

DTP_FEATURES = ['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4', 'dtp_avg', 'dtp_wavg']
FIN_FEATURES = ['credit_sale_amount', 'opening_balance',
                'amount_due_cumsum', 'amount_paid_cumsum']
ALL_FEATURES = DTP_FEATURES + FIN_FEATURES + ['payment_ratio']

BRACKET_ORDER  = ['30_days', '60_days', '90_days']  # on_time excluded
BRACKET_COLORS = {
    'on_time': '#2ecc71',
    '30_days': '#5dade2',
    '60_days': '#f0b27a',
    '90_days': '#e07b72',
}


# ─────────────────────────────────────────────
# DATA HELPERS
# ─────────────────────────────────────────────

def load_and_prepare(df):
    """Clean and split into X (float64) and y (bracket label)."""
    cols_needed = ALL_FEATURES + [BRACKET_COL]
    df_clean = df[cols_needed].dropna()
    df_clean = df_clean[df_clean[BRACKET_COL].isin(BRACKET_ORDER)]
    X = df_clean[ALL_FEATURES].astype(float).copy()
    y = df_clean[BRACKET_COL].copy()
    return X, y


def log1p_transform(X):
    """log1p-transform financial features; leave DTP features untouched."""
    X_t = X.astype(float).copy()
    for col in FIN_FEATURES:
        if col in X_t.columns:
            shift = abs(X_t[col].min()) if X_t[col].min() < 0 else 0
            X_t[col] = np.log1p(X_t[col] + shift)
    return X_t


def fit_lda(X, y):
    """Scale then fit LDA; return (pipeline, projected array, explained var ratio)."""
    n_components = min(len(BRACKET_ORDER) - 1, X.shape[1])   # max 3
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lda',    LinearDiscriminantAnalysis(n_components=n_components,
                                              solver='svd')),
    ])
    X_lda = pipe.fit_transform(X, y)
    evr   = pipe.named_steps['lda'].explained_variance_ratio_
    return pipe, X_lda, evr


def compute_class_separation(X, y):
    """Fisher between/within variance ratio per feature."""
    rows = []
    for col in X.columns:
        overall_mean = X[col].mean()
        classes      = y.unique()
        between = sum(
            (y == c).sum() * (X.loc[y == c, col].mean() - overall_mean) ** 2
            for c in classes
        ) / len(y)
        within = sum(
            X.loc[y == c, col].var() * (y == c).sum()
            for c in classes
        ) / len(y)
        rows.append({'feature': col,
                     'separation': between / within if within > 0 else 0})
    return pd.DataFrame(rows).sort_values('separation', ascending=False)


# ─────────────────────────────────────────────
# ATOMIC PLOT HELPERS
# ─────────────────────────────────────────────

def _draw_raw_hist(ax, col, y, feat, show_legend=False):
    """Overlapping density histogram for one raw feature."""
    col = col.astype(float)
    lo, hi = col.quantile(0.01), col.quantile(0.99)
    for bracket in BRACKET_ORDER:
        vals = col[y == bracket].clip(lo, hi)
        ax.hist(vals, bins=60, density=True, alpha=0.55,
                color=BRACKET_COLORS[bracket], label=bracket)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.tick_params(labelsize=7)
    if show_legend:
        ax.legend(fontsize=7, loc='upper right')


def _draw_ld_hist(ax, X_lda, y, evr, ld_idx, show_legend=False):
    """Density histogram for one LDA discriminant axis."""
    for bracket in BRACKET_ORDER:
        mask = (y == bracket).values
        ax.hist(X_lda[mask, ld_idx], bins=80, density=True, alpha=0.55,
                color=BRACKET_COLORS[bracket], label=bracket)
    ax.set_title(f'LD{ld_idx+1}  ({evr[ld_idx]*100:.1f}% sep. var.)',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel(f'LD{ld_idx+1} score', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.tick_params(labelsize=7)
    if show_legend:
        ax.legend(fontsize=7, loc='upper right')


def _draw_ld_scatter(ax, X_lda, y, evr):
    """LD1 vs LD2 2-D scatter coloured by bracket."""
    for bracket in BRACKET_ORDER:
        mask = (y == bracket).values
        ax.scatter(X_lda[mask, 0], X_lda[mask, 1],
                   s=5, alpha=0.28, color=BRACKET_COLORS[bracket],
                   label=bracket, rasterized=True)
    ax.set_xlabel(f'LD1  ({evr[0]*100:.1f}%)', fontsize=8)
    ld2_label = f'LD2  ({evr[1]*100:.1f}%)' if len(evr) > 1 else 'LD2'
    ax.set_ylabel(ld2_label, fontsize=8)
    ax.set_title('LD1 vs LD2 — 2-D projection', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7, markerscale=3)
    ax.tick_params(labelsize=7)


def _col_header(ax, text, color):
    """Solid-background header label above an axis."""
    ax.annotate(text, xy=(0.5, 1.24), xycoords='axes fraction',
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round,pad=0.4', fc=color,
                          alpha=0.85, ec='none'))


def _draw_separation_bar(ax, sep_before, sep_after):
    """Grouped bar chart: Fisher separation before vs after."""
    features = sep_before['feature'].tolist()
    x, w     = np.arange(len(features)), 0.35
    b_vals   = sep_before.set_index('feature').loc[features, 'separation'].values
    a_vals   = sep_after.set_index('feature').loc[features,  'separation'].values
    ax.bar(x - w/2, b_vals, w, label='Before (raw)',         color='#aab7b8')
    ax.bar(x + w/2, a_vals, w, label='After (log1p+scaled)', color='#5dade2')
    ax.set_xticks(x)
    ax.set_xticklabels(features, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel("Fisher's separation ratio", fontsize=9)
    ax.set_title("Per-feature class separation — before vs after log1p transform",
                 fontsize=10)
    ax.legend(fontsize=8)


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def run_lda_analysis_delinquent(df):
    """
    Two-column layout — one feature per row:

        LEFT  column  =  BEFORE  (raw histogram, p1–p99 clipped)
        RIGHT column  =  AFTER   (LDA component histogram)

    Feature rows:
        Rows 0–5   : dtp_1, dtp_2, dtp_3, dtp_4, dtp_avg, dtp_wavg
        Rows 6–10  : credit_sale_amount, opening_balance,
                     amount_due_cumsum, amount_paid_cumsum, payment_ratio

    Extra rows:
        Row 11 : [note | LD1 vs LD2 scatter]
        Row 12 : Fisher separation bar  (spans both columns)
    """
    # ── data ───────────────────────────────────
    X_raw, y         = load_and_prepare(df)
    X_t              = log1p_transform(X_raw)
    pipe, X_lda, evr = fit_lda(X_t, y)

    # console report
    print(f"\n{'='*55}")
    print("  LDA — Explained separation variance")
    print(f"{'='*55}")
    for i, v in enumerate(evr, 1):
        print(f"  LD{i}: {'█'*int(v*40):<40}  {v*100:.1f}%")
    print(f"{'='*55}\n")

    sep_before = compute_class_separation(X_raw, y)
    sep_after  = compute_class_separation(X_t,   y)
    merged = (sep_before.rename(columns={'separation': 'before'})
                         .merge(sep_after.rename(columns={'separation': 'after'}),
                                on='feature'))
    merged['improvement_%'] = ((merged['after'] - merged['before'])
                               / merged['before'].replace(0, np.nan) * 100)
    print(merged.to_string(index=False))

    # ── grid ───────────────────────────────────
    fin_plus  = FIN_FEATURES + ['payment_ratio']    # 5 features
    all_feats = DTP_FEATURES + fin_plus             # 11 total
    n_feats   = len(all_feats)
    n_ld      = X_lda.shape[1]                      # 3 at most

    # Right column content (after):
    #   Row 0        → LD1 histogram
    #   Row 1        → LD2 histogram
    #   Row 2        → LD3 histogram
    #   Row 3        → LD1 vs LD2 scatter  (spans remaining right cells via axis('off'))
    #   Rows 4–10    → empty (right side blank, left keeps all raw features)
    # Bottom row     → Fisher bar spanning both columns

    NROWS = n_feats + 1    # 11 feature rows + 1 bar row
    NCOLS = 2

    fig, axes = plt.subplots(
        NROWS, NCOLS,
        figsize=(14, NROWS * 2.6),
        gridspec_kw={'hspace': 0.80, 'wspace': 0.28}
    )
    fig.suptitle(
        "LDA Analysis — Delinquent Only (30 / 60 / 90 days)  |  on_time excluded",
        fontsize=13, fontweight='bold', y=1.001
    )

    # column header badges on the first row
    _col_header(axes[0, 0], "◀  BEFORE  —  Raw feature distributions", '#922b21')
    _col_header(axes[0, 1], "AFTER  —  LDA discriminant components  ▶", '#1a5276')

    # ── LEFT column: all 11 raw feature histograms ──
    for i, feat in enumerate(all_feats):
        _draw_raw_hist(axes[i, 0], X_raw[feat], y, feat,
                       show_legend=(i == 0))

    # ── RIGHT column ────────────────────────────────
    # Rows 0–(n_ld-1): one LD histogram each
    for ld_idx in range(n_ld):
        _draw_ld_hist(axes[ld_idx, 1], X_lda, y, evr, ld_idx,
                      show_legend=(ld_idx == 0))

    # Row n_ld: LD1 vs LD2 scatter
    _draw_ld_scatter(axes[n_ld, 1], X_lda, y, evr)

    # Rows (n_ld+1) onward on the right: turn off (left keeps raw hists)
    for i in range(n_ld + 1, n_feats):
        axes[i, 1].axis('off')

    # ── Fisher bar row — spans both columns ─────────
    br = n_feats
    for c in range(NCOLS):
        axes[br, c].axis('off')

    pos_l  = axes[br, 0].get_position()
    pos_r  = axes[br, 1].get_position()
    bar_ax = fig.add_axes((
        pos_l.x0,
        pos_l.y0,
        pos_r.x1 - pos_l.x0,
        pos_l.height * 1.8
    ))
    _draw_separation_bar(bar_ax, sep_before, sep_after)

    plt.savefig('lda_delinquent_only.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nSaved → lda_delinquent_only.png")

    return pipe, X_lda, evr, merged


# ─────────────────────────────────────────────
# USAGE
# ─────────────────────────────────────────────
# pipe, X_lda, evr, sep_df = run_lda_analysis(df_credit_sales)  # delinquent only
#
# Inject LD components back into your dataframe for modelling:
#   X_raw, y = load_and_prepare(df_credit_sales)
#   X_t      = log1p_transform(X_raw)
#   comps    = pipe.transform(X_t)
#   for i in range(comps.shape[1]):
#       df_credit_sales.loc[X_raw.index, f'LD{i+1}'] = comps[:, i]

In [ ]:
run_lda_analysis(df_credit_sales)

# D. Machine Learning Pipelines

## 1. Single model trainer

### a. Data preparation

In [ ]:
# ── 1. Load settings ────────────────────────────────────────────────────────────────────────
from datetime import datetime
from types import SimpleNamespace
from utils.data_loaders.read_settings_json import read_settings_json

settings = read_settings_json(file_path="settings.json")
observation_end = datetime.strptime(settings['Training']['observation_end'], "%Y/%m/%d")
target_feature = settings["Training"]["target_feature"]
test_size = float(settings["Training"]["test_size"])

# SimpleNamespace is picklable by multiprocessing workers (class defined in __main__ is not).
args = SimpleNamespace(
    observation_end = observation_end,
    target_feature  = target_feature,
    test_size       = test_size,                                          # Test size in %
    parameters_dir  = settings['Training']['MODEL_PARAMETERS'],
)



# ── 2. Load the invoice dataset ─────────────────────────────────────────────────────────
from feature_engineering.credit_sales_machine_learning import CreditSalesProcessor

cs = CreditSalesProcessor(
    df_revenues, df_enrollees, args,
    drop_demographic_columns=True,
    drop_fully_paid_invoices=False,
    drop_helper_columns=True,
    drop_missing_dtp=True,
    add_streak_features=True,
    exclude_school_years=[2016, 2017, 2018],
    winsorise_dtp=True)
df_credit_sales = cs.show_data()



# ── 3. Load the dataset for machine learning and for survival analysis ───────────────────
survival_columns = ['days_elapsed_until_fully_paid', 'censor']
non_survival_columns = ['due_date', 'dtp_bracket']


df_data = df_credit_sales[df_credit_sales['censor'] == 1].copy()
df_data.drop(columns=survival_columns, inplace=True)

df_data_surv = df_credit_sales.drop(columns=non_survival_columns)



# ── 4. Cox Best Parameters (hardcoded from prior tuning run) ──────────────────────────────
from machine_learning.utils.features.adjust_survival_time_periods import adjust_payment_period
from machine_learning.utils.features.get_slope_time_points import get_slope_timepoints

X_surv = df_data_surv.drop(columns=survival_columns)
T      = adjust_payment_period(df_data_surv["days_elapsed_until_fully_paid"])   
E      = df_data_surv["censor"]

# Best parameters confirmed by prior tuning run (C-index: 0.7817)
# Hardcoded to skip the 90-fit CV sweep and save ~3–5 minutes per run
best_surv_parameters = {"alpha": 0.05, "l1_ratio": 0.5}
best_time_points = get_slope_timepoints(T, E, n_points=9)
args.time_points = best_time_points

class _TunerStub:
    best_params_  = best_surv_parameters
    best_c_index_ = 0.7817

tuner = _TunerStub()



# ── 5. Data Preparation ──────────────────────────────────────────────────────────────
from machine_learning.utils.data.data_preparation import DataPreparer

preparer = DataPreparer(
    df_data,
    target_feature=args.target_feature,
    test_size=args.test_size
)
preparer.encode_labels().train_test_split().resample(balance_strategy="smote_tomek")

X_train = preparer.X_train
X_test = preparer.X_test
y_train = preparer.y_train
y_test = preparer.y_test



# ── 6. Generate survival features ──────────────────────────────────────────────────────────────
from machine_learning.utils.features.generate_survival_features import generate_survival_features

X_survival_train, X_survival_test = generate_survival_features(
    X_surv, T, E, X_train, X_test,
    best_params=best_surv_parameters,
    time_points=best_time_points
)


### b. Train models (Ada Boost)

In [ ]:
model_parameters = {
    "learning_rate": 0.1,
    "n_estimators": 50,
    "random_state": 42
}

In [ ]:
from machine_learning import AdaBoostPipeline

pipeline = AdaBoostPipeline(
    X_train, X_test, y_train, y_test,
    args,
    model_parameters
)

# Capture results from pipeline
result = pipeline.initialize_model().fit(use_feature_selection=True).evaluate().show_results()
selected_features = pipeline.features

print(f'Accuracy: {result['accuracy']}')
print(f'Precission: {result['precision_macro']}')
print(f'Recall: {result['recall_macro']}')
print(f'F1: {result['f1_macro']}')
print(f'AUC: {result['roc_auc_macro']}')

In [ ]:
selected_features.weights

### b. Train models (Two Step)

In [ ]:
model_parameters = {
    "stage1": {
        "colsample_bytree": 0.8,
        "learning_rate": 0.01,
        "max_depth": 3,
        "min_child_weight": 3,
        "n_estimators": 300,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "subsample": 0.8
    },
    "stage2": {
        "learning_rate": 0.1,
        "n_estimators": 50
    }
}

In [ ]:
from machine_learning import TwoStagePipeline
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier

pipeline = TwoStagePipeline(
    X_survival_train, X_survival_test, y_train, y_test,
    args,
    stage1_estimator=XGBClassifier(**model_parameters["stage1"]),
    stage2_estimator=AdaBoostClassifier(**model_parameters["stage2"]),
    use_lda = [True, False],
    lda_mode = "append",
)

result = pipeline.initialize_model().fit(use_feature_selection=True).evaluate().show_results()
selected_features = pipeline.features

In [ ]:
result.keys()

In [ ]:
print(f'Accuracy: {result['accuracy']}')
print(f'Precission: {result['precision_macro']}')
print(f'Recall: {result['recall_macro']}')
print(f'F1: {result['f1_macro']}')
print(f'AUC: {result['roc_auc_macro']}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Confusion matrix (convert to NumPy array)
cm = np.array(result['confusion_matrix'])

# Class mapping
class_mapping = {'on_time': 0, '30_days': 1, '60_days': 2, '90_days': 3}
labels = list(class_mapping.keys())

# Normalize by row (true class)
cm_normalized = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

# Create annotations with both raw counts and percentages
annot = np.empty_like(cm).astype(str)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot[i, j] = f"{cm[i, j]}\n({cm_normalized[i, j]:.1%})"

# Plot heatmap using normalized values for color
plt.figure(figsize=(8,6))
sns.heatmap(cm_normalized, annot=annot, fmt="", cmap="Blues",
            xticklabels=labels, yticklabels=labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Counts + Percentages)")
plt.show()

In [ ]:
selected_features

In [ ]:
assert selected_features.weights is not None
df = pd.DataFrame.from_dict(selected_features.weights['stage_1'], orient='index', columns=['value'])
df['value'].sum()

## 2. Multiple model trainer

In [ ]:
from machine_learning import (
    AdaBoostPipeline,
    DecisionTreePipeline,
    GaussianNaiveBayesPipeline,
    KNearestNeighborPipeline,
    RandomForestPipeline,
    XGBoostPipeline,
    OrdinalPipeline,
    TwoStagePipeline,
)

models = {
    "ada_boost":           AdaBoostPipeline,
    "decision_tree":       DecisionTreePipeline,
    "gaussian_naive_bayes": GaussianNaiveBayesPipeline,
    "knn":                 KNearestNeighborPipeline,
    "random_forest":       RandomForestPipeline,
    "xgboost":             XGBoostPipeline,
    "ordinal":             OrdinalPipeline,    # expands to ordinal_xgboost, ordinal_random_forest, ordinal_ada_boost
    "two_stage":           TwoStagePipeline,   # expands to 6 xgb/rf/ada combinations
}

# XGBoost uses GPU acceleration — must run sequentially to avoid CUDA context
# conflicts in the thread pool. The runner auto-extends this list to include
# all expanded model names containing 'xgb' (ordinal_xgboost, two_stage_xgb_*,
# two_stage_ada_xgb), so only the base alias is needed here.
do_not_parallel_compute = ['xgboost']


In [ ]:
from machine_learning.utils.features.generate_thresholds import generate_thresholds

# Full strategy matrix: no balancing baseline + all SMOTE variants + hybrid threshold sweep
# smote_enn removed — prior runs showed it consistently underperforms borderline_smote
balance_strategies = ["none", "smote", "borderline_smote", "smote_tomek", "hybrid"]

# Hybrid threshold grid: three representative thresholds (threshold choice is a second-order effect)
thresholds = [0.5, 0.7, 0.9]


In [ ]:
# To silence the error when running knn:
# UserWarning: Could not find the number of physical cores for the following reason:
# [WinError 2]
import os

os.environ['OMP_NUM_THREADS'] = '16'

In [ ]:
# (Single-threaded runner removed — parallel runner in SurvivalExperimentRunner is the only path)


In [ ]:
# Survival related features
drop_columns = ['censor', 'days_elapsed_until_fully_paid']

# Only extract invoices with payments
df_data = df_credit_sales[df_credit_sales['censor'] == 1]

df_data = df_data.drop(columns=drop_columns)

# Drop invoices with missing critical features
df_data.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'], inplace=True)
df_data

In [ ]:
import time
from machine_learning.utils.training.run_models_parallel import SurvivalExperimentRunner
from machine_learning.utils.io.save_results_to_folder import save_training_results

# Record start time
_train_start = time.time()
_train_start_iso = time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime(_train_start))

# Create an experiment runner instance
runner = SurvivalExperimentRunner(
    df_data=df_data,
    df_data_surv=df_data_surv,
    models=models,
    balance_strategies=balance_strategies,
    args=args,
    best_parameters=best_surv_parameters,
    thresholds=thresholds,
    n_jobs=-1,
    do_not_parallel_compute=do_not_parallel_compute,
    feature_selection_baseline=True,
    feature_selection_enhanced=True,
)

# Run all experiments — returns (results_df, class_mappings)
df_results, class_mappings = runner.run()

# Record end time
_train_end = time.time()
_train_end_iso = time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime(_train_end))
_elapsed_s = int(_train_end - _train_start)
_elapsed_str = f"{_elapsed_s // 3600}h {(_elapsed_s % 3600) // 60}m {_elapsed_s % 60}s"
print(f"\nTotal training time: {_elapsed_str}")

# Save results to a new dated folder under Results/
survival_results_dict = {
    "best_c_index":      tuner.best_c_index_,
    "best_parameters":   best_surv_parameters,
    "time_points":       best_time_points,
}

metadata, run_folder = save_training_results(
    model_results_df     = df_results,
    survival_results_dict= survival_results_dict,
    class_mappings_dict  = class_mappings,
    base_output_folder   = settings['Training']['RESULTS_ROOT'],
    model_names          = list(models.keys()),
    start_time           = _train_start_iso,
    end_time             = _train_end_iso,
    total_run_time       = _elapsed_str,
    format               = "sqlite",
)

print(f"Results saved → {run_folder}")


In [ ]:
df_results

In [ ]:
df_results.sort_values(by='enhanced_accuracy', ascending=False)

In [ ]:
df_results.sort_values(by='enhanced_precision_macro', ascending=False)

In [ ]:
df_results.sort_values(by='enhanced_f1_macro', ascending=False)

In [ ]:
df_results.sort_values(by='enhanced_roc_auc_macro', ascending=False)

In [ ]:
import pandas as pd

# File path
file_path = r"C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\MachineLearning\Results\06 model_results - no feature selection.xlsx"

# Read the Excel file
df = pd.read_excel(file_path)

# Choose which score column to use
score_column = "enhanced_roc_auc_macro"   # <-- change this to the metric you want (e.g., "f1", "roc_auc", etc.)

# Create a combined column: balance_strategy + threshold (only for hybrid)
df["balance_strategy_full"] = df.apply(
    lambda row: f"{row['balance_strategy']}_{row['undersample_threshold']}"
    if row["balance_strategy"] == "hybrid" else row["balance_strategy"],
    axis=1
)

# Compute mean score per model+parameters+strategy
grouped = (
    df.groupby(["model", "parameters", "balance_strategy_full"])[score_column]
    .mean()
    .reset_index()
)

# Rank strategies within each model+parameters group
grouped["rank"] = grouped.groupby(["model", "parameters"])[score_column] \
                         .rank(method="first", ascending=False)

# Assign weighted points: top 1 → 5, top 2 → 4, … top 5 → 1
def assign_points(rank):
    if rank == 1: return 5
    elif rank == 2: return 4
    elif rank == 3: return 3
    elif rank == 4: return 2
    elif rank == 5: return 1
    else: return 0

grouped["points"] = grouped["rank"].apply(assign_points)

# Aggregate across all models to see total points per strategy
strategy_scores = (
    grouped.groupby("balance_strategy_full")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)

# Show results
print("Weighted ranking per model+parameters:")
print(grouped.sort_values(["model", "parameters", "rank"]))

print("\nGlobal tally of weighted points per balance strategy:")
print(strategy_scores)

In [ ]:
import pandas as pd

# File path
file_path = r"C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\Results\08 - model_results - fixed enhancement features (with feature selection).xlsx"

# Read the Excel file
df = pd.read_excel(file_path)

# Choose which score column to use
score_column = "enhanced_f1_macro"   # <-- change this to the metric you want (e.g., "f1", "roc_auc", etc.)

# Create a combined column: balance_strategy + threshold (only for hybrid)
df["balance_strategy_full"] = df.apply(
    lambda row: f"{row['balance_strategy']}_{row['undersample_threshold']}"
    if row["balance_strategy"] == "hybrid" else row["balance_strategy"],
    axis=1
)

# Compute mean score per model+parameters+strategy
grouped = (
    df.groupby(["model", "parameters", "balance_strategy_full"])[score_column]
    .mean()
    .reset_index()
)

# Rank strategies within each model+parameters group
grouped["rank"] = grouped.groupby(["model", "parameters"])[score_column] \
                         .rank(method="first", ascending=False)

# Assign weighted points: top 1 → 5, top 2 → 4, … top 5 → 1
def assign_points(rank):
    if rank == 1: return 5
    elif rank == 2: return 4
    elif rank == 3: return 3
    elif rank == 4: return 2
    elif rank == 5: return 1
    else: return 0

grouped["points"] = grouped["rank"].apply(assign_points)

# Aggregate across all models to see total points per strategy
strategy_scores = (
    grouped.groupby("balance_strategy_full")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)

# Show results
print("Weighted ranking per model+parameters:")
print(grouped.sort_values(["model", "parameters", "rank"]))

print("\nGlobal tally of weighted points per balance strategy:")
print(strategy_scores)

## 3. Forecasting pipeline

In [ ]:
from feature_engineering.credit_sales_machine_learning import CreditSalesProcessor

cs_test = CreditSalesProcessor(df_revenues, df_enrollees, args,
                      drop_fully_paid_invoices=True,
                      drop_back_account_transactions=True,
                      calculate_payment_amounts=True,
                      add_description=True,
                      drop_missing_dtp=False)
df_cs_test = cs_test.show_data()
df_cs_test